### Transformer

<img src="diagrams/Transformer_Architecture.png" width="500px">

#### Imports

In [ ]:
import torch
import torch.nn as nn
import math

#### Input embedding layer

<img src="diagrams/InputLayer.png" width="1000px">

<img src="diagrams/InputEmbedding.png" width="1000px">

In [2]:
class InputEmbedding(nn.Module):
     def __init__(self, 
                  vocabulary_size: int, 
                  dimensionE: int) -> None:
          super().__init__()

          self.dimensionE = dimensionE
          self.embedding = nn.Embedding(vocabulary_size, dimensionE)

     def forward(self, tokenIDs):
          embeddingStrength = math.sqrt(self.dimensionE)
          return self.embedding(tokenIDs) * embeddingStrength

In [3]:
tokenIDs = torch.tensor([2, 43, 21])
ie = InputEmbedding(vocabulary_size=200, dimensionE=6)(tokenIDs)
ie

tensor([[-2.7344, -2.1616,  2.5998, -2.0297, -3.5991, -1.5002],
        [-2.9016, -0.7896,  1.3867, -0.3961,  2.0090,  1.1009],
        [-1.1796, -3.0920,  3.0058, -0.9485,  3.4533,  1.5097]],
       grad_fn=<MulBackward0>)

#### Positional encoding

<img src="diagrams/PositionalEncoding.png" width="1000px">

In [4]:
class PositionalEncoding(nn.Module):
     def __init__(self, 
                  sequenceLength: int ,
                  dimensionE: int, 
                  dropout: float) -> None:
          super().__init__()

          self.sequenceLength = sequenceLength
          self.dimensionE = dimensionE
          self.dropout = nn.Dropout(dropout)

          peMatrix = torch.zeros(sequenceLength, dimensionE)
          posVector = torch.arange(0, sequenceLength, dtype=torch.float).unsqueeze(1)
          frequencyCreator = torch.exp(torch.arange(0, dimensionE, 2).float() * (-math.log(10000.0) / dimensionE))

          peMatrix[:, 0::2] = torch.sin(posVector * frequencyCreator)
          peMatrix[:, 1::2] = torch.cos(posVector * frequencyCreator)

          peMatrix = peMatrix.unsqueeze(0)    # (1, sequenceLength, dimensionE)

          self.register_buffer("peMatrix", peMatrix)

     def forward(self, inputEmbedding):
          inputEmbedding = inputEmbedding + self.peMatrix[: , :inputEmbedding.shape[1], :].requires_grad_(False)
          return self.dropout(inputEmbedding)

In [5]:
PositionalEncoding(sequenceLength=3, dimensionE=6, dropout=0.1)(ie.unsqueeze(0))

tensor([[[-3.0382, -0.0000,  2.8886, -1.1442, -3.9990, -0.5558],
         [-2.2890, -0.2770,  0.0000,  0.6698,  2.2346,  2.3343],
         [-0.3003, -3.8980,  3.4427,  0.0525,  3.8418,  2.7885]]],
       grad_fn=<MulBackward0>)

#### Multi Head Attention

<img src="diagrams/MultiHeadAttention1.png" width="1000px"> 

<img src="diagrams/MultiHeadAttention2.png" width="800px">

<img src="diagrams/masking.png" width="700px">

In [6]:
class MultiHeadAttentionBlock(nn.Module):
     def __init__(self, 
                  heads: int, 
                  dropout: float, 
                  dimensionE: int) -> None:
          super().__init__()

          self.dimensionE = dimensionE
          self.dropout = nn.Dropout(dropout)
          self.heads = heads
          assert dimensionE % heads == 0, "Embedding vector must be equally divided among attention heads"
          self.sizeHeadVector = dimensionE // heads
          self.wQ = nn.Linear(dimensionE, dimensionE, bias=False)
          self.wK = nn.Linear(dimensionE, dimensionE, bias=False)
          self.wV = nn.Linear(dimensionE, dimensionE, bias=False)
          self.wO = nn.Linear(dimensionE, dimensionE, bias=False)

     @staticmethod
     def attention(query, key, value, mask, dropout: nn.Dropout):
          dimensionKey = key.shape[-1]
          attentionValues = query @ key.transpose(-2, -1)
          attentionValues /= math.sqrt(dimensionKey)
          if mask is not None:
               attentionValues = attentionValues.masked_fill(mask==0, -1e9)
          attentionValues = attentionValues.softmax(dim=-1)
          if dropout is not None:
               attentionValues = dropout(attentionValues)
          return attentionValues @ value, attentionValues
     
     def forward(self, q, k, v, mask):
          # (batch, sequenceLength, embedding vector size)
          query = self.wQ(q)         
          key = self.wK(k)
          value = self.wV(v)

          #  (batch size, sequenceLength, heads, embedding vector size)     to     (batch size, heads, sequenceLength, embedding vector size)
          query = query.view(query.shape[0], query.shape[1], self.heads, self.sizeHeadVector)      
          query = query.transpose(1, 2)            

          key = key.view(key.shape[0], key.shape[1], self.heads, self.sizeHeadVector)      
          key = key.transpose(1, 2)

          value = value.view(value.shape[0], value.shape[1], self.heads, self.sizeHeadVector)      
          value = value.transpose(1, 2)      

          attentionQKV, attentionValues = MultiHeadAttentionBlock.attention(
               query = query,
               key = key,
               value = value,
               mask = mask,
               dropout = self.dropout
          )

          attentionQKV = attentionQKV.transpose(1,2)
          attentionQKV = attentionQKV.contiguous()
          attentionQKV = attentionQKV.view(attentionQKV.shape[0], -1, self.heads * self.sizeHeadVector)      # batch, sequenceLength, dimensionE

          return self.wO(attentionQKV)

#### Layer Normalisation

In [7]:
class LayerNormalisation(nn.Module):
     def __init__(self, 
                  dimensionE: int, 
                  microValue: float = 10**-7) -> None:
          super().__init__()

          self.dimensionE = dimensionE
          self.microValue = microValue
          self.gamma = nn.Parameter(torch.ones(dimensionE))
          self.beta = nn.Parameter(torch.zeros(dimensionE))

     def forward(self, input):
          mean = input.mean(dim=-1, keepdim=True)
          std = input.std(dim=-1, keepdim=True)
          normalisedInput = (input - mean) / (std + self.microValue)

          return self.gamma * normalisedInput + self.beta

#### Feed forward block

<img src="diagrams/feedForward.png" width="600px">

In [8]:
class FeedForwardBlock(nn.Module):
     def __init__(self, 
                  dimensionE: int, 
                  neurons: int, 
                  dropout: float) -> None:
          super().__init__()

          self.dimensionE = dimensionE
          self.neurons = neurons
          self.layer1 = nn.Linear(dimensionE, neurons)
          self.dropout = nn.Dropout(dropout)
          self.layer2 = nn.Linear(neurons, dimensionE)

     def forward(self, input):
          return self.layer2(self.dropout(torch.relu((self.layer1(input)))))

#### Residual Connection

<img src="diagrams/residualConnection.png" width = "400px">

In [9]:
class ResidualConnection(nn.Module):
     def __init__(self, 
                  dimensionE: int, 
                  dropout: float) -> None:
          super().__init__()
          self.dropout = nn.Dropout(dropout)
          self.normalise = LayerNormalisation(dimensionE)

     def forward(self, input, sublayer):
          connection = self.dropout(sublayer(self.normalise(input))) + input
          return connection

#### Encoder block

<img src="diagrams/encoderBlock.png" width="200px">

In [10]:
class EncoderBlock(nn.Module):
     def __init__(self, 
                  dimensionE: int, 
                  multiHeadAttention: MultiHeadAttentionBlock, 
                  feedForward: FeedForwardBlock) -> None:
          super().__init__()

          self.multiHeadAttention = multiHeadAttention
          self.feedForward = feedForward
          self.residualConnections = nn.ModuleList([ResidualConnection(dimensionE, dropout) for _ in range(2)])

     def forward(self, input, mask):
          input = self.residualConnections[0](input, lambda input: self.multiHeadAttention(input, input, input, mask))
          input = self.residualConnections[1](input, self.feedForward)
          return input

#### Encoder

In [11]:
class Encoder(nn.Module):
     def __init__(self, 
                  dimensionE: int, 
                  layers: nn.ModuleList) -> None:
          super().__init__()
          self.dimensionE = dimensionE
          self.layers = layers
          self.normalise = LayerNormalisation(dimensionE)

     def forward(self, input, mask):
          for encoderBlock in self.layers:
               input = encoderBlock(input, mask)
          return self.normalise(input)

#### Decoder block

<img src="diagrams/decoderBlock.png" width="200px">

In [12]:
class DecoderBlock(nn.Module):
     def __init__(self, 
                  dimensionE: int, 
                  maskedMultiHeadAttention: MultiHeadAttentionBlock, 
                  multiHeadAttention: MultiHeadAttentionBlock, 
                  feedForward: FeedForwardBlock, 
                  dropout: float) -> None:
          super().__init__()

          self.maskedMultiHeadAttention = maskedMultiHeadAttention
          self.multiHeadAttention = multiHeadAttention
          self.feedForward = feedForward
          self.residualConnections = nn.ModuleList([ResidualConnection(dimensionE, dropout) for _ in range(3)])

     def forward(self, input, encoderOutput, sourceMask, targetMask):
          # source mask - hide tokens like <pad>
          # target mask - hide future tokens
          input = self.residualConnections[0](input, lambda input: self.maskedMultiHeadAttention(input, input, input, targetMask))
          input = self.residualConnections[1](input, lambda input: self.multiHeadAttention(input, encoderOutput, encoderOutput, sourceMask))
          input = self.residualConnections[2](input, self.feedForward)

          return input

#### Decoder

In [13]:
class Decoder(nn.Module):
     def __init__(self, 
                  dimesnionE: int, 
                  layers: nn.ModuleList) -> None:
          super().__init__()
          self.layers = layers
          self.normalise = LayerNormalisation(dimesnionE)

     def forward(self, input, encoderOutput, sourceMask, targetMask):
          for decoderBlock in self.layers:
               input = decoderBlock(input, encoderOutput, sourceMask, targetMask)
          return self.normalise(input)

#### Projection layer

In [14]:
class ProjectionLayer(nn.Module):
     def __init__(self, 
                  dimesnionE, 
                  vocabularySize) -> None:
          super().__init__()
          self.layer = nn.Linear(dimesnionE, vocabularySize)
     
     def forward(self, input):
          return self.layer(input)

#### Transformer (combining everything)

In [15]:
class Transformer(nn.Module):
     def __init__(self, 
                  encoder: Encoder, 
                  decoder: Decoder, 
                  projectionLayer: ProjectionLayer,
                  sourceEmbedding: InputEmbedding,
                  targetEmbedding: InputEmbedding,
                  sourcePE: PositionalEncoding,
                  targetPE: PositionalEncoding) -> None:
          super().__init__()
        
          self.encoder = encoder
          self.decoder = decoder
          self.projectionLayer = projectionLayer
          self.sourceEmbedding = sourceEmbedding
          self.targetEmbedding = targetEmbedding
          self.sourcePE = sourcePE
          self.targetPE = targetPE

     def encode(self, input, mask):
          input = self.sourceEmbedding(input)
          input = self.sourcePE(input)
          return self.encoder(input, mask)
     
     def decode(self, target: torch.Tensor, encoderOutput: torch.Tensor, sourceMask: torch.Tensor, targetMask: torch.tensor):
          target = self.targetEmbedding(target)
          target = self.targetPE(target)
          return self.decoder(target, encoderOutput, sourceMask, targetMask)
     
     def project(self, input):
          return self.projectionLayer(input)

In [16]:
def buildTransformer(
          sourceVocabSize: int,
          targetVocabSize: int,
          sourceSequenceLength: int,
          targetSequenceLength: int,
          dimensionE: int = 512,
          Nx: int = 6,
          heads: int = 8,
          dropout: float = 0.1,
          neurons: int = 2048) -> Transformer:
     
     sourceEmbedding = InputEmbedding(sourceVocabSize, dimensionE)
     targetEmbedding = InputEmbedding(targetVocabSize, dimensionE)

     sourcePE = PositionalEncoding(sourceSequenceLength, dimensionE, dropout)
     targetPE = PositionalEncoding(targetSequenceLength, dimensionE, dropout)

     encoderBlocks = []
     for _ in range(Nx):
          encoderBlock = EncoderBlock(
               dimensionE= dimensionE,
               multiHeadAttention=MultiHeadAttentionBlock(heads, dropout, dimensionE),
               feedForward=FeedForwardBlock(dimensionE, neurons, dropout),
          )
          encoderBlocks.append(encoderBlock)

     decoderBlocks = []
     for _ in range(Nx):
          decoderBlock = DecoderBlock(
               dimensionE=dimensionE,
               maskedMultiHeadAttention=MultiHeadAttentionBlock(heads, dropout, dimensionE),
               multiHeadAttention=MultiHeadAttentionBlock(heads, dropout, dimensionE),
               feedForward=FeedForwardBlock(dimensionE, neurons, dropout),
               dropout=dropout
          )
          decoderBlocks.append(decoderBlock)
     
     encoder = Encoder(dimensionE, nn.ModuleList(encoderBlocks))
     decoder = Decoder(dimensionE, nn.ModuleList(decoderBlocks))

     projectionLayer = ProjectionLayer(dimensionE, targetVocabSize)

     transformer = Transformer(
          encoder=encoder,
          decoder=decoder,
          projectionLayer=projectionLayer,
          sourceEmbedding=sourceEmbedding,
          targetEmbedding=targetEmbedding,
          sourcePE=sourcePE,
          targetPE=targetPE
     )

     for param in transformer.parameters():
          if param.dim() > 1:
               nn.init.xavier_uniform(param)
               
     return transformer

#### Dataset

In [17]:
from torch.utils.data import Dataset

In [37]:
def createMask(size: int):
     mask = torch.tril(torch.ones((1, size, size)))
     return mask.bool()

In [38]:
mask = createMask(5)
mask

tensor([[[ True, False, False, False, False],
         [ True,  True, False, False, False],
         [ True,  True,  True, False, False],
         [ True,  True,  True,  True, False],
         [ True,  True,  True,  True,  True]]])

In [34]:
class TranslationalDataset(Dataset):
     def __init__(self,
                  pairs,
                  sourceVocabulary,
                  targetVocabulary,
                  sequenceLength: int=10):
          super().__init__()

          self.pairs = pairs
          self.sourceVocabulary = sourceVocabulary
          self.targetVocabulary = targetVocabulary
          self.sequenceLength = sequenceLength

          self.source_pad_id = sourceVocabulary["[PAD]"]
          self.source_sos_id = sourceVocabulary["[SOS]"]
          self.source_eos_id = sourceVocabulary["[EOS]"]
          self.source_unk_id = sourceVocabulary["[UNK]"]

          self.target_pad_id = targetVocabulary["[PAD]"]
          self.target_sos_id = targetVocabulary["[SOS]"]
          self.target_eos_id = targetVocabulary["[EOS]"]
          self.target_unk_id = targetVocabulary["[UNK]"]

     def __len__(self):
          return len(self.pairs)
     
     def encodeSentence(self, sentence, vocabulary):
          tokens = sentence.split()
          tokenIds = [
               vocabulary.get(token, vocabulary["[UNK]"])
               for token in tokens
          ]
          return tokenIds
     
     def padSequence(self, tokenIds, pad_id):
          paddingTokens = self.sequenceLength - len(tokenIds)
          paddedTokens = tokenIds + [pad_id] * paddingTokens
          return paddedTokens
     
     def __getitem__(self, id):
          sourceText, targetText = self.pairs[id]

          sourceTokens = self.encodeSentence(sourceText, self.sourceVocabulary)
          targetTokens = self.encodeSentence(targetText, self.targetVocabulary)

          encoderInput = [self.source_sos_id] + sourceTokens + [self.source_eos_id]
          decoderInput = [self.target_sos_id] + targetTokens
          label = targetTokens + [self.target_eos_id]

          encoderInput = self.padSequence(encoderInput, self.source_pad_id)
          decoderInput = self.padSequence(decoderInput, self.target_pad_id)
          label = self.padSequence(label, self.target_pad_id)

          encoderInput = torch.tensor(encoderInput, dtype=torch.long)
          decoderInput = torch.tensor(decoderInput, dtype=torch.long)
          label = torch.tensor(label, dtype=torch.long)

          encoderMask = (
               encoderInput != self.source_pad_id
          ).unsqueeze(0).unsqueeze(0)

          decoderPadMask = (
               decoderInput != self.target_pad_id
          ).unsqueeze(0).unsqueeze(0)

          decoderHideMask = createMask(self.sequenceLength)
          decoderMask = decoderPadMask & decoderHideMask

          return {
            "encoder_input": encoderInput,      
            "decoder_input": decoderInput,      
            "encoder_mask": encoderMask,        
            "decoder_mask": decoderMask,        
            "label": label,                  
            "source_text": sourceText,
            "target_text": targetText,
        }

In [32]:
pairs = [
    ("i like cats", "j aime chats"),
    ("i like dogs", "j aime chiens"),
    ("you like cats", "tu aime chats"),
    ("you like dogs", "tu aime chiens"),
]

source_vocab = {
    "[PAD]":   0,
    "[SOS]":   1,
    "[EOS]":   2,
    "[UNK]":   3,
    "i":            4,
    "you":       5,
    "like":       6,
    "cats":      7,
    "dogs":    8,
}

target_vocab = {
    "[PAD]":     0,
    "[SOS]":     1,
    "[EOS]":     2,
    "[UNK]":     3,
    "j":              4,
    "tu":            5,
    "aime":       6,
    "chats":      7,
    "chiens":    8,
}

In [35]:
dataset = TranslationalDataset(
     pairs=pairs,
     sourceVocabulary=source_vocab,
     targetVocabulary=target_vocab,
     sequenceLength=8
)

In [39]:
sample = dataset[0]
sample

{'encoder_input': tensor([1, 4, 6, 7, 2, 0, 0, 0]),
 'decoder_input': tensor([1, 4, 6, 7, 0, 0, 0, 0]),
 'encoder_mask': tensor([[[ True,  True,  True,  True,  True, False, False, False]]]),
 'decoder_mask': tensor([[[ True, False, False, False, False, False, False, False],
          [ True,  True, False, False, False, False, False, False],
          [ True,  True,  True, False, False, False, False, False],
          [ True,  True,  True,  True, False, False, False, False],
          [ True,  True,  True,  True, False, False, False, False],
          [ True,  True,  True,  True, False, False, False, False],
          [ True,  True,  True,  True, False, False, False, False],
          [ True,  True,  True,  True, False, False, False, False]]]),
 'label': tensor([4, 6, 7, 2, 0, 0, 0, 0]),
 'source_text': 'i like cats',
 'target_text': 'j aime chats'}